# ema-first-moment — worked example 1: First-moment EMA on one buffer

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `ema-first-moment`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Adam's first moment is an exponentially weighted moving average of the gradient: `m = beta1*m + (1-beta1)*g`. The `(1-beta1)` factor is what distinguishes it from plain momentum — it caps the terminal velocity so the EMA converges to the mean gradient rather than accumulating without bound. The buffer is mutated in place with `copy_` so its identity survives across steps.

## Worked solution

We run the first-moment update on a single buffer over several steps with known gradients.

1. Start with a zero first-moment buffer `m`. We capture its `data_ptr()` so we can prove later that the buffer object never gets reallocated.
2. Each step we have a gradient `g`. The new EMA value is `beta1*m + (1-beta1)*g`: a fraction `beta1` of the old estimate plus a fraction `(1-beta1)` of the fresh gradient.
3. We write that back with `m.copy_(...)` rather than `m = ...`. The in-place copy keeps the same storage, which is what Adam needs so the optimizer state stays attached to the right parameter.
4. Because the first moment preserves sign, feeding alternating-sign gradients makes `m` track their running average — it does not square anything (that is the second moment).

In [ ]:
import torch as t

t.manual_seed(0)
beta1 = 0.9
m = t.zeros(3)
grads = [t.tensor([1.0, -2.0, 0.5]), t.tensor([1.0, -2.0, 0.5]), t.tensor([1.0, -2.0, 0.5])]

def ema_m_step(m, g, beta1):
    m.copy_(beta1 * m + (1 - beta1) * g)
    return m

ptr = m.data_ptr()
for g in grads:
    ema_m_step(m, g, beta1)
print('m:', m.tolist())
print('in-place preserved:', m.data_ptr() == ptr)